# Lab 03 — Dự báo phụ tải điện theo chuỗi thời gian

Notebook thực hành dự báo một giờ tiếp theo cho chuỗi phụ tải AEP (`AEP_MW`). Notebook dùng dữ liệu thật `data/raw/AEP_hourly.csv` và các bộ ước lượng NumPy tự cài đặt trong `src/lab03`; không tự chuyển sang dữ liệu giả nếu thiếu file.

**Quy trình:** nạp/làm sạch dữ liệu → tạo đặc trưng chỉ từ quá khứ → chia Train/Validation/Test theo thời gian → walk-forward CV trên Train → chọn mô hình bằng Validation → đánh giá một lần trên Test → dự báo đệ quy 24 giờ.

> Chạy lần lượt từ trên xuống bằng **Run All**. Không chạy ngược các cell phụ thuộc biến ở cell trước.

## Mục tiêu và hợp đồng dự báo

Sau bài lab, người học có thể: (1) biến chuỗi giờ thành bảng supervised mà không dùng tương lai; (2) chia dữ liệu theo thời gian; (3) phân biệt CV, Validation và Test; (4) cài đặt/so sánh baseline, hồi quy tuyến tính và mô hình cây; (5) diễn giải sai số và lưu artifact tái lập được.

Tại thời điểm dự báo `t`, mục tiêu là tải điện giờ kế tiếp `y_t`; mô hình chỉ được nhận dữ liệu quan sát đến `t-1` cùng biến lịch đã biết trước. Bài dùng **dự báo một bước cuốn chiếu** để đánh giá Test, và **dự báo đệ quy 24 bước tại một mốc cố định** cho tương lai. Hai kết quả trả lời hai câu hỏi vận hành khác nhau.

## Bối cảnh và từ điển dữ liệu

| Trường | Ý nghĩa | Dùng trong notebook |
| --- | --- | --- |
| `Datetime` | Timestamp theo giờ của quan sát | Index thời gian; thứ tự thời gian không bị xáo trộn |
| `AEP_MW` | Công suất/phụ tải AEP, đơn vị MW | Biến mục tiêu đơn biến |

Nguồn có timestamp không kèm timezone. Các giờ DST lặp được phân biệt bằng timestamp trùng và loader giữ bản ghi đầu tiên; giờ thiếu nội bộ được nội suy tối đa ba giờ. Đây là quyết định xử lý dữ liệu có thể ảnh hưởng các mốc DST, không phải khôi phục chính xác timezone gốc.

## 0. Chuẩn bị môi trường

Chạy tại thư mục gốc repository. Nếu chưa cài môi trường, mở terminal tại thư mục gốc và chạy `python -m pip install -r requirements.txt`, sau đó `jupyter lab` hoặc `jupyter notebook`.

In [ ]:
from datetime import datetime
from pathlib import Path
import json
import pickle
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "lab03").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Không tìm thấy src/lab03. Hãy mở notebook trong repository Lab_TimeSeries.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lab03.data import load_hourly_series, chronological_split
from src.lab03.features import build_supervised_features, recursive_forecast
from src.lab03.cv import evaluate_walk_forward
from src.lab03.metrics import regression_metrics
from src.lab03.lab03_utils.report import write_report_bundle
from src.lab03.lab03_models import (
    NaiveLastValue, LinearRegressionScratch, DecisionTreeRegressorScratch,
    RandomForestRegressorScratch, XGBoostScratch,
)

DATA_PATH = ROOT / "data" / "raw" / "AEP_hourly.csv"
if not DATA_PATH.is_file():
    raise FileNotFoundError(f"Thiếu dữ liệu AEP: {DATA_PATH}. Đặt AEP_hourly.csv vào data/raw/ rồi chạy lại.")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ROOT / "reports" / "lab03" / "notebook_runs" / RUN_ID
FIGURE_DIR = RUN_DIR / "figures"
MODEL_DIR = RUN_DIR / "models"
for directory in (RUN_DIR, FIGURE_DIR, MODEL_DIR):
    directory.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
print("Project root:", ROOT)
print("Run ID:", RUN_ID)
print("Dataset:", DATA_PATH)

## 1. Nạp và kiểm tra dữ liệu

Dữ liệu được sắp xếp theo thời gian, loại timestamp trùng, đưa về lưới một giờ và chỉ nội suy các khoảng trống nội bộ tối đa ba giờ. Khoảng trống dài hơn sẽ báo lỗi để tránh tự tạo dữ liệu không có căn cứ.

In [ ]:
raw_preview = pd.read_csv(DATA_PATH)
required_columns = {"Datetime", "AEP_MW"}
missing_columns = required_columns.difference(raw_preview.columns)
if missing_columns:
    raise ValueError(f"Thiếu cột dữ liệu bắt buộc: {sorted(missing_columns)}")
print("Shape gốc:", raw_preview.shape)
print("Cột:", list(raw_preview.columns))
display(raw_preview.head())
raw_quality = pd.Series({
    "rows": len(raw_preview),
    "duplicate timestamps": raw_preview["Datetime"].duplicated().sum(),
    "invalid/missing timestamps": pd.to_datetime(raw_preview["Datetime"], errors="coerce").isna().sum(),
    "missing targets": raw_preview["AEP_MW"].isna().sum(),
    "input already time-sorted": pd.to_datetime(raw_preview["Datetime"], errors="coerce").is_monotonic_increasing,
}, name="raw value").to_frame()
display(raw_quality)

In [ ]:
series, ingestion = load_hourly_series(DATA_PATH)
display(pd.Series(ingestion, name="value").to_frame())
display(series.describe().to_frame("AEP_MW").round(2))
assert isinstance(series.index, pd.DatetimeIndex)
assert series.index.is_monotonic_increasing and series.index.is_unique
assert series.notna().all()
print(f"Dữ liệu sạch: {len(series):,} giờ | {series.index.min()} → {series.index.max()}")

### Đọc audit nạp và làm sạch

`raw_rows` là kích thước đầu vào; `rows_after_drop_invalid` loại dòng không đọc được; duplicate timestamps bị loại trước khi lập lưới giờ. `missing_periods_inserted` đếm timestamp được thêm vào lưới regular trước nội suy. Số hàng sau làm sạch có thể tăng vì các mốc giờ thiếu được chèn.

In [ ]:
cleaning_audit = pd.DataFrame([
    {"check": "Rows in raw CSV", "value": ingestion["raw_rows"]},
    {"check": "Rows after invalid rows removed", "value": ingestion["rows_after_drop_invalid"]},
    {"check": "Duplicate timestamps removed", "value": ingestion["duplicate_timestamps_removed"]},
    {"check": "Missing hourly periods inserted", "value": ingestion["missing_periods_inserted"]},
    {"check": "Rows after regularization", "value": len(series)},
    {"check": "Missing values remaining", "value": int(series.isna().sum())},
])
display(cleaning_audit)
assert len(series) == ingestion["observations_after_cleaning"]
assert series.notna().all()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
series.plot(ax=axes[0], linewidth=0.45, title="AEP hourly load — toàn bộ chuỗi")
axes[0].set_ylabel("AEP_MW")
series.loc["2016-01-04":"2016-01-17"].plot(ax=axes[1], linewidth=1.0, title="Hai tuần minh họa")
axes[1].set_ylabel("AEP_MW")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "01_hourly_series.png", dpi=140, bbox_inches="tight")
plt.show()

## 1.1 Khám phá chu kỳ trong dữ liệu

Chuỗi phụ tải theo giờ thường có nhịp ngày và khác biệt giữa ngày làm việc/cuối tuần. Các profile dưới đây là thống kê mô tả trên toàn chuỗi để hiểu dữ liệu; chúng **không** được dùng để chọn mô hình hoặc tính điểm Test. Calendar features tuần hoàn trong mô hình được tạo theo timestamp, không cần nhìn giá trị tương lai.

In [ ]:
hour_profile = series.groupby(series.index.hour).agg(["mean", "std"])
weekday_profile = series.groupby(series.index.dayofweek).agg(["mean", "std"])
weekday_profile.index = [pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(d)) for d in weekday_profile.index]
weekday_profile.index = [date.day_name() for date in weekday_profile.index]
hour_weekday = pd.DataFrame({"load": series.to_numpy(), "hour": series.index.hour,
                             "weekday": series.index.dayofweek}).pivot_table(
    index="weekday", columns="hour", values="load", aggfunc="mean"
)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(hour_profile.index, hour_profile["mean"], marker="o", markersize=3)
axes[0].fill_between(hour_profile.index,
                     hour_profile["mean"] - hour_profile["std"],
                     hour_profile["mean"] + hour_profile["std"], alpha=0.2)
axes[0].set(title="Profile theo giờ trong ngày", xlabel="Giờ", ylabel="AEP_MW")
axes[1].bar(weekday_profile.index, weekday_profile["mean"],
            yerr=weekday_profile["std"], capsize=2)
axes[1].set(title="Profile theo thứ", ylabel="Mean AEP_MW")
axes[1].tick_params(axis="x", rotation=35)
im = axes[2].imshow(hour_weekday.to_numpy(), aspect="auto", origin="lower", cmap="viridis")
axes[2].set(title="Mean load: thứ × giờ", xlabel="Giờ", ylabel="Thứ trong tuần")
axes[2].set_yticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
fig.colorbar(im, ax=axes[2], label="AEP_MW")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "06_temporal_profiles.png", dpi=140, bbox_inches="tight")
plt.show()
display(hour_profile.round(1).head(6))
display(weekday_profile.round(1))

In [ ]:
daily_load = series.resample("D").mean()
rolling_30d = daily_load.rolling(30, min_periods=15).mean()
fig, ax = plt.subplots(figsize=(13, 4))
daily_load.plot(ax=ax, linewidth=0.45, alpha=0.65, label="Daily mean")
rolling_30d.plot(ax=ax, linewidth=1.5, label="30-day rolling mean")
ax.set_title("Phụ tải trung bình ngày và xu hướng làm mượt")
ax.set_ylabel("AEP_MW")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "07_daily_trend.png", dpi=140, bbox_inches="tight")
plt.show()

## 2. Tạo đặc trưng dự báo một bước

Đặc trưng gồm các lag 1–168 giờ, trung bình/độ lệch chuẩn trượt 6–168 giờ và lịch tuần hoàn. Hàm tạo rolling dùng `shift(1)`, do đó không lấy giá trị mục tiêu tại thời điểm dự báo.

Với mục tiêu tại `t`, `lag_k(t) = y_(t-k)`. Ví dụ `lag_24` là tải cùng giờ ngày trước; rolling mean 24 giờ là trung bình của `y_(t-24), ..., y_(t-1)`. Không dùng `y_t` trong bất kỳ lag/rolling feature nào. Sin/cos mã hóa tính tuần hoàn để giờ 23 gần giờ 0 hơn so với mã hóa số nguyên.

In [ ]:
LAGS = (1, 2, 3, 6, 12, 24, 48, 168)
ROLLING_WINDOWS = (6, 12, 24, 168)
X, y = build_supervised_features(
    series, lags=LAGS, rolling_windows=ROLLING_WINDOWS, include_calendar=True
)
display(X.head(3).round(2))
print("X:", X.shape, "y:", y.shape, "| số đặc trưng:", X.shape[1])

timestamp = X.index[200]
position = series.index.get_loc(timestamp)
assert X.loc[timestamp, "lag_1"] == series.iloc[position - 1]
assert np.isclose(X.loc[timestamp, "roll_mean_6"], series.iloc[position - 6:position].mean())
assert y.loc[timestamp] == series.loc[timestamp]
print("Kiểm tra causal lag/rolling: đạt.")

### Kiểm tra một hàng supervised bằng tay

Các hàng đầu tiên không có đủ lịch sử cho lag 168 hoặc rolling 168 nên được loại tự động. Bảng dưới đây đối chiếu một timestamp cụ thể với những giá trị đã biết trước thời điểm mục tiêu; kiểm tra này giúp phát hiện lệch index hoặc vô tình dùng target hiện tại.

In [ ]:
example_timestamp = X.index[200]
example_columns = ["lag_1", "lag_24", "lag_168", "roll_mean_24", "roll_std_24",
                   "hour_sin", "hour_cos", "dow_sin", "dow_cos"]
example_row = X.loc[[example_timestamp], example_columns].T.rename(columns={example_timestamp: "feature value"})
example_row.loc["target y_t"] = y.loc[example_timestamp]
display(example_row.round(3))
print("Prediction target time:", example_timestamp,
      "| latest lag value time:", example_timestamp - pd.Timedelta(hours=1))

## 3. Chia Train / Validation / Test theo thời gian

Tỷ lệ 70/15/15, không shuffle. Validation dùng chọn mô hình; Test được giữ riêng đến bước đánh giá cuối.

Features được tạo trước khi chia vì mỗi feature tại `t` chỉ phụ thuộc quá khứ. Do đó vài lag của Validation có thể lấy lịch sử cuối Train và lag đầu Test có thể lấy lịch sử cuối Validation; đây là ngữ cảnh đã biết tại thời điểm dự báo, không phải dùng target tương lai. Chỉ nên dùng cho giao thức dự báo cuốn chiếu được mô tả ở phần đánh giá.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = chronological_split(
    X, y, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15
)
split_summary = pd.DataFrame([
    {"split": name, "rows": len(target), "start": target.index.min(),
     "end": target.index.max(), "target_mean": target.mean()}
    for name, target in (("Train", y_train), ("Validation", y_val), ("Test", y_test))
]).set_index("split")
display(split_summary)
assert X_train.index.max() < X_val.index.min() < X_test.index.min()

fig, ax = plt.subplots(figsize=(13, 3.5))
for name, target in (("Train", y_train), ("Validation", y_val), ("Test", y_test)):
    ax.plot(target.index, target, linewidth=0.5, label=name)
ax.set_title("Chia tập theo thứ tự thời gian")
ax.set_ylabel("AEP_MW")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "02_temporal_split.png", dpi=140, bbox_inches="tight")
plt.show()

## 4. Walk-forward cross-validation trên Train

Năm fold mở rộng theo thời gian, chỉ dùng Train. Ở mỗi fold, model fit trên prefix cũ và chấm trên block kế tiếp. Feature của block validation được tạo theo giao thức cuốn chiếu: khi đã quan sát một giờ thì giá trị đó có thể làm lag cho giờ tiếp theo. CV vì thế không phải dự báo cố định nhiều ngày từ một origin.

Các cấu hình cây được giới hạn vừa phải để notebook chạy được trên máy cá nhân; đây là cấu hình thực hành, không phải tuning sản xuất.

### Các ứng viên và vai trò của chúng

| Mô hình | Cách hoạt động | Vai trò / giới hạn |
| --- | --- | --- |
| Naive Last-Value | Dự báo bằng `lag_1` | Baseline khó đánh bại ở chuỗi có quán tính |
| Linear Regression | OLS giải bằng NumPy, có ridge ổn định hóa nhỏ | Baseline tuyến tính nhanh; khó mô tả quan hệ phi tuyến |
| CART | Chọn ngưỡng chia giảm SSE bằng lưới quantile | Dễ diễn giải; một cây có phương sai cao |
| Random Forest | Trung bình nhiều cây CART bootstrap, random feature subsets | Giảm phương sai; tốn thời gian và có giới hạn số dòng |
| XGBoost-style | Boosting cây histogram với gradient/Hessian và regularization | Bắt tương tác phi tuyến; bản học thuật NumPy, không phải thư viện XGBoost chính thức |

Bốn mô hình học tham số được cài đặt trong `src/lab03/lab03_models/`; notebook không dùng estimator của scikit-learn hoặc XGBoost.

In [ ]:
MODEL_FACTORIES = {
    "Naive Last-Value": lambda: NaiveLastValue(),
    "Linear Regression": lambda: LinearRegressionScratch(ridge=1e-6),
    "CART": lambda: DecisionTreeRegressorScratch(
        max_depth=5, min_samples_split=30, min_samples_leaf=15,
        max_thresholds=8, random_state=42, max_train_rows=2500),
    "Random Forest": lambda: RandomForestRegressorScratch(
        n_estimators=5, max_depth=5, min_samples_split=30, min_samples_leaf=15,
        max_features="sqrt", max_thresholds=8, random_state=42, max_train_rows=2500),
    "XGBoost-style": lambda: XGBoostScratch(
        n_estimators=12, learning_rate=0.06, max_depth=3, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8, max_bins=12,
        random_state=42, max_train_rows=5000),
}

cv_detail, cv_aggregate = evaluate_walk_forward(
    X_train, y_train, MODEL_FACTORIES, n_splits=5
)
display(cv_detail.groupby("model")["RMSE"].agg(["mean", "std"]).sort_values("mean").round(2))
display(cv_detail.pivot(index="fold", columns="model", values="RMSE").round(2))

In [ ]:
cv_rmse_pivot = cv_detail.pivot(index="fold", columns="model", values="RMSE")
fig, ax = plt.subplots(figsize=(11, 4.5))
cv_rmse_pivot.plot(ax=ax, marker="o")
ax.set_title("Walk-forward CV: RMSE theo fold (Train only)")
ax.set_xlabel("Fold tiến về phía trước")
ax.set_ylabel("RMSE (MW)")
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "08_walk_forward_cv.png", dpi=140, bbox_inches="tight")
plt.show()
display(cv_detail.groupby("model").agg(
    folds=("fold", "nunique"), mean_train_rows=("train_rows", "mean"),
    mean_validation_rows=("validation_rows", "mean"),
    mean_RMSE=("RMSE", "mean"), std_RMSE=("RMSE", "std"),
).sort_values("mean_RMSE").round(2))

## 5. Chọn mô hình bằng Validation

Mỗi ứng viên được fit trên Train và chấm trên cùng một Validation. Chốt champion bằng `argmin(validation_RMSE)` trước khi xem Test. RMSE tính bằng căn bậc hai của trung bình bình phương residual; đơn vị vẫn là MW và phạt lỗi lớn mạnh hơn MAE.

In [ ]:
validation_rows = []
for name, factory in MODEL_FACTORIES.items():
    model = factory()
    fit_start = time.perf_counter()
    model.fit(X_train.to_numpy(dtype=float), y_train.to_numpy(dtype=float))
    fit_seconds = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    prediction = model.predict(X_val.to_numpy(dtype=float))
    predict_seconds = time.perf_counter() - predict_start
    validation_rows.append({"model": name, **regression_metrics(y_val, prediction),
                            "fit_seconds": fit_seconds, "predict_seconds": predict_seconds})
validation_summary = pd.DataFrame(validation_rows).set_index("model").sort_values("RMSE")
display(validation_summary.round(3))
CHAMPION_NAME = str(validation_summary.index[0])
print("Champion theo Validation RMSE:", CHAMPION_NAME)

### Cấu hình dùng cho lần chạy này

Các tham số được đặt trước để có thể so sánh trong một lần chạy; đây không phải kết quả hyperparameter search. `max_train_rows` giới hạn dữ liệu đưa vào cây/boosting và được ghi trong summary để không che khuất khác biệt so với Linear Regression.

In [ ]:
parameter_rows = []
for name, factory in MODEL_FACTORIES.items():
    params = factory().get_params()
    parameter_rows.append({"model": name, "parameters": json.dumps(params, sort_keys=True)})
display(pd.DataFrame(parameter_rows).set_index("model"))
print("Rows in Train:", len(X_train), "| full candidate fit rows (Train+Validation):", len(X_train) + len(X_val))

## 6. Đánh giá trên Test (không dùng để chọn mô hình)

Fit lại mỗi ứng viên trên Train + Validation rồi đánh giá trên Test. Bảng Test chỉ dùng báo cáo hiệu năng; không thay đổi champion đã chốt. Với protocol one-step rolling, feature ở các timestamp Test sau có thể dùng giá trị thực đã quan sát tại timestamp Test trước.

In [ ]:
X_train_val = pd.concat([X_train, X_val])
y_train_val = pd.concat([y_train, y_val])
test_predictions = {}
final_models = {}
final_fit_seconds = {}
test_predict_seconds = {}
test_rows = []
for name, factory in MODEL_FACTORIES.items():
    model = factory()
    fit_start = time.perf_counter()
    model.fit(X_train_val.to_numpy(dtype=float), y_train_val.to_numpy(dtype=float))
    final_fit_seconds[name] = time.perf_counter() - fit_start
    predict_start = time.perf_counter()
    prediction = np.asarray(model.predict(X_test.to_numpy(dtype=float)), dtype=float)
    test_predict_seconds[name] = time.perf_counter() - predict_start
    final_models[name] = model
    test_predictions[name] = prediction
    test_rows.append({"model": name, **regression_metrics(y_test, prediction)})
test_summary = pd.DataFrame(test_rows).set_index("model")
test_summary.insert(0, "validation_RMSE", validation_summary["RMSE"])
display(test_summary.sort_values("validation_RMSE").round(3))
print("Champion vẫn được xác định từ Validation:", CHAMPION_NAME)

### Đọc bảng Test đúng cách

Các metric Test là đánh giá cuối cùng cho ứng viên được fit trên Train + Validation. `MAPE` là phần trăm và có thể không ổn định nếu target gần 0; `R²` không phải accuracy và có thể âm. Mô hình có RMSE nhỏ nhất trên Test không tự động trở thành champion—chọn champion chỉ bằng Validation để tránh tối ưu trực tiếp trên Test.

In [ ]:
baseline_name = "Naive Last-Value"
baseline_test_rmse = float(test_summary.loc[baseline_name, "RMSE"])
comparison = test_summary[["validation_RMSE", "MAE", "RMSE", "MAPE (%)", "R2"]].copy()
comparison["RMSE improvement vs naive (%)"] = (
    (baseline_test_rmse - comparison["RMSE"]) / baseline_test_rmse * 100
)
display(comparison.sort_values("validation_RMSE").round(3))
print("Champion selection remains:", CHAMPION_NAME,
      "(minimum Validation RMSE, not minimum Test RMSE).")

In [ ]:
n_show = min(24 * 7, len(y_test))
fig, axes = plt.subplots(2, 1, figsize=(13, 7))
axes[0].plot(y_test.index[:n_show], y_test.iloc[:n_show], label="Actual", linewidth=1.5)
axes[0].plot(y_test.index[:n_show], test_predictions[CHAMPION_NAME][:n_show],
             label=f"{CHAMPION_NAME} prediction", linewidth=1.1)
axes[0].set_title("Dự báo một bước — tuần đầu của Test")
axes[0].legend()
residual = y_test.to_numpy(dtype=float) - test_predictions[CHAMPION_NAME]
axes[1].plot(y_test.index, residual, linewidth=0.45)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set_title(f"Residual trên toàn Test — {CHAMPION_NAME}")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "03_test_and_residuals.png", dpi=140, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
test_summary.sort_values("RMSE")["RMSE"].plot.bar(ax=ax)
ax.set_title("So sánh RMSE trên tập Test (thấp hơn tốt hơn)")
ax.set_ylabel("RMSE (MW)")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "04_test_model_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

### Phân tích residual và sai số theo thời điểm

Residual được định nghĩa là `prediction - actual`; bias dương nghĩa là mô hình dự báo cao. Nhóm sai số theo giờ và thứ giúp phát hiện mô hình yếu ở một phần chu kỳ ngày/tuần mà một RMSE tổng thể có thể che mất. Đây là phân tích sau khi đã khóa model, không dùng để quay lại chọn mô hình trên Test.

In [ ]:
test_diagnostics = pd.DataFrame(index=y_test.index)
test_diagnostics["actual"] = y_test.to_numpy(dtype=float)
test_diagnostics["prediction"] = test_predictions[CHAMPION_NAME]
test_diagnostics["error"] = test_diagnostics["prediction"] - test_diagnostics["actual"]
test_diagnostics["absolute_error"] = test_diagnostics["error"].abs()
test_diagnostics["hour"] = test_diagnostics.index.hour
test_diagnostics["weekday"] = test_diagnostics.index.dayofweek
error_by_hour = test_diagnostics.groupby("hour").agg(
    MAE=("absolute_error", "mean"), bias=("error", "mean"))
error_by_weekday = test_diagnostics.groupby("weekday").agg(
    MAE=("absolute_error", "mean"), bias=("error", "mean"))
display(error_by_hour.round(2))
display(error_by_weekday.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
error_by_hour["MAE"].plot(ax=axes[0], marker="o", markersize=3)
axes[0].set(title="Test MAE theo giờ", xlabel="Giờ", ylabel="MAE (MW)")
error_by_hour["bias"].plot(ax=axes[1], marker="o", markersize=3)
axes[1].axhline(0, color="black", linewidth=0.8)
axes[1].set(title="Test bias theo giờ", xlabel="Giờ", ylabel="Prediction − actual (MW)")
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "09_test_error_profiles.png", dpi=140, bbox_inches="tight")
plt.show()
print("Overall residual mean:", round(float(test_diagnostics["error"].mean()), 2), "MW")
print("Overall residual std:", round(float(test_diagnostics["error"].std()), 2), "MW")

### Biểu đồ chẩn đoán bổ sung cho báo cáo

Biểu đồ phân tán kiểm tra mức độ bám của dự báo so với giá trị thực. Đường chéo đỏ là dự báo hoàn hảo `prediction = actual`. Histogram residual cho biết sai số tập trung ở đâu và mô hình có xu hướng dự báo thấp hay cao. Hai biểu đồ này nên được đọc cùng với bảng MAE/bias theo giờ.

In [ ]:
actual = test_diagnostics["actual"]
prediction = test_diagnostics["prediction"]
residual = test_diagnostics["error"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
axes[0].scatter(actual, prediction, s=8, alpha=0.28, edgecolors="none")
limits = [min(actual.min(), prediction.min()), max(actual.max(), prediction.max())]
axes[0].plot(limits, limits, color="tab:red", linestyle="--", linewidth=1.5,
             label="Prediction = actual")
axes[0].set(title=f"Actual vs prediction — {CHAMPION_NAME}",
            xlabel="Actual AEP_MW", ylabel="Predicted AEP_MW")
axes[0].legend()

axes[1].hist(residual, bins=50, color="tab:blue", alpha=0.82,
             edgecolor="white", linewidth=0.25)
axes[1].axvline(0, color="black", linewidth=1.0)
axes[1].axvline(residual.mean(), color="tab:red", linestyle="--", linewidth=1.5,
                label=f"Mean bias: {residual.mean():.1f} MW")
axes[1].set(title="Phân phối residual", xlabel="Prediction − actual (MW)",
            ylabel="Số quan sát")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "11_actual_vs_prediction.png", dpi=140, bbox_inches="tight")
plt.show()

weekday_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
error_heatmap = test_diagnostics.pivot_table(
    index="weekday", columns="hour", values="absolute_error", aggfunc="mean"
).reindex(index=range(7), columns=range(24))
fig, ax = plt.subplots(figsize=(13, 5))
im = ax.imshow(error_heatmap.to_numpy(), aspect="auto", cmap="YlOrRd", origin="upper")
ax.set(title="MAE theo thứ và giờ trên Test", xlabel="Giờ trong ngày", ylabel="Thứ")
ax.set_xticks(range(24), range(24))
ax.set_yticks(range(7), weekday_names)
fig.colorbar(im, ax=ax, label="MAE (MW)")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "12_test_mae_weekday_hour.png", dpi=140, bbox_inches="tight")
plt.show()

### Đặc trưng nào liên quan mạnh với dự báo?

Với mô hình tuyến tính, hiển thị trị tuyệt đối của hệ số nhân độ lệch chuẩn đặc trưng để đưa các feature về thang so sánh gần nhau hơn. Với cây, dùng split-gain importance. Cả hai chỉ mô tả cách mô hình sử dụng feature, không chứng minh quan hệ nhân quả; các lag thường tương quan mạnh với nhau.

In [ ]:
champion_model = final_models[CHAMPION_NAME]
feature_importance = getattr(champion_model, "feature_importances_", None)
if feature_importance is not None:
    effect_values = np.asarray(feature_importance, dtype=float)
    effect_label = "Normalized split gain"
else:
    coefficients = np.asarray(champion_model.coef_, dtype=float)
    feature_scale = np.asarray(champion_model.feature_scale_, dtype=float)
    effect_values = np.abs(coefficients * feature_scale)
    effect_label = "Absolute coefficient × feature standard deviation"
feature_effects = pd.DataFrame({"feature": X.columns, "effect": effect_values})
feature_effects = feature_effects.sort_values("effect", ascending=False)
display(feature_effects.head(15).round(4))
top_effects = feature_effects.head(15).sort_values("effect")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_effects["feature"], top_effects["effect"])
ax.set_title(f"Top feature effects — {CHAMPION_NAME}")
ax.set_xlabel(effect_label)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "10_feature_effects.png", dpi=140, bbox_inches="tight")
plt.show()

## 7. Dự báo đệ quy 24 giờ tiếp theo

Đây là dự báo tại một mốc cố định: mô hình champion được fit lại trên toàn bộ dữ liệu có nhãn, sau đó các dự báo được đưa ngược vào lịch sử cho bước kế tiếp. Không dùng giá trị thực tương lai.

In [ ]:
forecast_model = MODEL_FACTORIES[CHAMPION_NAME]()
forecast_model.fit(X.to_numpy(dtype=float), y.to_numpy(dtype=float))
forecast = recursive_forecast(
    forecast_model, series, horizon=24, feature_names=X.columns,
    lags=LAGS, rolling_windows=ROLLING_WINDOWS, include_calendar=True
)
display(forecast.to_frame("forecast_AEP_MW").round(2))
assert len(forecast) == 24 and forecast.index.min() > series.index.max()

fig, ax = plt.subplots(figsize=(12, 4.5))
series.iloc[-24 * 7:].plot(ax=ax, label="Observed", linewidth=1.3)
forecast.plot(ax=ax, label="24-hour recursive forecast", marker="o", linewidth=1.2)
ax.set_title(f"Dự báo 24 giờ — champion: {CHAMPION_NAME}")
ax.set_ylabel("AEP_MW")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "05_future_24h_forecast.png", dpi=140, bbox_inches="tight")
plt.show()

### Diễn giải dự báo và giới hạn

Mô hình cho ra một quỹ đạo điểm 24 giờ, không phải khoảng dự báo. Sai số có thể tích lũy vì mỗi bước sau dùng dự báo bước trước làm lag; khoảng bất định thường tăng theo horizon. Mô hình được fit trên các hàng supervised có nhãn khả dụng đến cuối chuỗi, sau khi đã hoàn tất đánh giá Test. Do đó dự báo này không phải backtest độc lập, cũng không cung cấp bảo đảm về ngày tương lai có chế độ vận hành khác.

In [ ]:
forecast_summary = forecast.describe().rename("AEP_MW").to_frame().round(2)
display(forecast_summary)
print("Forecast origin:", series.index.max())
print("Forecast horizon:", len(forecast), "hours")
print("First-to-last forecast change:", round(float(forecast.iloc[-1] - forecast.iloc[0]), 2), "MW")

## 8. Tạo report và model summary

Mỗi lần chạy lưu đầy đủ kết quả trong `reports/lab03/notebook_runs/<run_id>/` và cập nhật các bản mới nhất tại `reports/lab03/` cùng `reports/model_summary/lab03/`. Bộ report gồm Markdown, HTML, PDF; model summary gồm CSV, JSON, Markdown và các bảng metric.

In [ ]:
REPORT_ROOT = ROOT / "reports" / "lab03"
SUMMARY_ROOT = ROOT / "reports" / "model_summary" / "lab03"
REPORT_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

model_files = {}
for name, model in final_models.items():
    slug = name.lower().replace(" ", "_").replace("-", "_")
    model_path = MODEL_DIR / f"{slug}.pkl"
    with model_path.open("wb") as file:
        pickle.dump(model, file)
    model_files[name] = model_path
with (MODEL_DIR / "champion_future_forecast.pkl").open("wb") as file:
    pickle.dump(forecast_model, file)

series.rename("AEP_MW").to_csv(RUN_DIR / "cleaned_hourly_series.csv", header=True)
cv_detail.to_csv(RUN_DIR / "cv_fold_metrics.csv", index=False)
validation_summary.to_csv(RUN_DIR / "validation_metrics.csv")
test_summary.to_csv(RUN_DIR / "test_metrics.csv")
pd.DataFrame({"actual": y_test, **test_predictions}).to_csv(RUN_DIR / "test_predictions.csv")
forecast.to_csv(RUN_DIR / "future_24h_forecast.csv", header=True)

metric_names = ["MAE", "MSE", "RMSE", "MAPE (%)", "sMAPE (%)", "R2"]
cv_stats = cv_detail.groupby("model")[metric_names].agg(["mean", "std"])
summary_rows = []
for name in MODEL_FACTORIES:
    row = {
        "run_id": RUN_ID,
        "model": name,
        "selected_champion": name == CHAMPION_NAME,
        "feature_count": int(X.shape[1]),
        "total_train_validation_rows": int(len(X_train_val)),
        "training_rows_used": int(getattr(final_models[name], "n_training_rows_", len(X_train_val))),
        "fit_seconds": float(final_fit_seconds[name]),
        "predict_seconds_test": float(test_predict_seconds[name]),
        "model_file": model_files[name].relative_to(ROOT).as_posix(),
        "model_parameters": json.dumps(final_models[name].get_params(), sort_keys=True),
    }
    for metric in metric_names:
        row[f"cv_{metric}_mean"] = float(cv_stats.loc[name, (metric, "mean")])
        row[f"cv_{metric}_std"] = float(cv_stats.loc[name, (metric, "std")])
        row[f"validation_{metric}"] = float(validation_summary.loc[name, metric])
        row[f"test_{metric}"] = float(test_summary.loc[name, metric])
    summary_rows.append(row)
model_summary = pd.DataFrame(summary_rows).sort_values("validation_RMSE")

summary_columns = [
    "run_id", "model", "selected_champion", "feature_count",
    "total_train_validation_rows", "training_rows_used", "fit_seconds",
    "predict_seconds_test", "model_file", "model_parameters",
    *[f"cv_{metric}_{stat}" for metric in metric_names for stat in ("mean", "std")],
    *[f"validation_{metric}" for metric in metric_names],
    *[f"test_{metric}" for metric in metric_names],
]
model_summary = model_summary[summary_columns]
model_summary_json = {
    "run_id": RUN_ID,
    "champion_selected_by": "minimum validation_RMSE",
    "champion": CHAMPION_NAME,
    "models": summary_rows,
}
validation_json = {
    "run_id": RUN_ID,
    "selection_metric": "RMSE",
    "models": validation_summary.reset_index().to_dict(orient="records"),
}
test_json = {
    "run_id": RUN_ID,
    "champion_selected_by_validation": CHAMPION_NAME,
    "models": test_summary.reset_index().to_dict(orient="records"),
}

def write_json(path, payload):
    with Path(path).open("w", encoding="utf-8") as file:
        json.dump(payload, file, ensure_ascii=False, indent=2,
                  default=lambda value: value.item() if isinstance(value, np.generic) else str(value),
                  allow_nan=False)

def write_model_summary(directory):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model_summary.to_csv(directory / "model_summary.csv", index=False)
    write_json(directory / "model_summary.json", model_summary_json)
    summary_view = model_summary.assign(
        champion_label=model_summary["selected_champion"].map({True: "Yes", False: "No"})
    )[[
        "model", "champion_label", "cv_RMSE_mean", "cv_RMSE_std",
        "validation_RMSE", "test_MAE", "test_RMSE", "test_MAPE (%)",
        "test_R2", "fit_seconds", "training_rows_used",
    ]].rename(columns={
        "model": "Model", "champion_label": "Champion",
        "cv_RMSE_mean": "CV RMSE mean", "cv_RMSE_std": "CV RMSE std",
        "validation_RMSE": "Validation RMSE", "test_MAE": "Test MAE",
        "test_RMSE": "Test RMSE", "test_MAPE (%)": "Test MAPE (%)",
        "test_R2": "Test R2", "fit_seconds": "Fit (s)",
        "training_rows_used": "Rows used",
    })
    standard_model_dir = ROOT / "models" / "lab03"
    standard_log_dir = ROOT / "logs" / "lab03"
    standard_models = sorted(standard_model_dir.glob("*.pkl")) if standard_model_dir.is_dir() else []
    run_logs = sorted(standard_log_dir.glob("*.log")) if standard_log_dir.is_dir() else []
    summary_lines = [
        "# Lab 03 Model Summary",
        "",
        f"**Run:** `{RUN_ID}`",
        f"**Dataset:** `{ingestion['dataset_name']}`",
        f"**Cleaned observations:** {len(series):,} hourly rows",
        f"**Features:** {X.shape[1]}",
        f"**Split:** Train {len(X_train):,} | Validation {len(X_val):,} | Test {len(X_test):,}",
        f"**Champion:** {CHAMPION_NAME}, selected by lowest Validation RMSE",
        "",
        "Test metrics below are held-out results; they were not used to select the champion.",
        "",
        "## Candidate comparison",
        "",
        "| " + " | ".join(str(column) for column in summary_view.columns) + " |",
        "| " + " | ".join("---" for _ in summary_view.columns) + " |",
    ]
    for values in summary_view.itertuples(index=False, name=None):
        cells = []
        for value in values:
            if pd.isna(value):
                cells.append("N/A")
            elif isinstance(value, (float, np.floating)):
                cells.append(f"{value:.3f}")
            else:
                cells.append(str(value).replace("|", "\\|"))
        summary_lines.append("| " + " | ".join(cells) + " |")
    summary_lines += [
        "",
        "CV uses five expanding-window folds within Train. Candidate models are fitted on Train + Validation before Test scoring.",
        "",
        "## Model files",
        "",
        f"Candidate models fitted on Train + Validation: `{(RUN_DIR / 'models').relative_to(ROOT).as_posix()}/`.",
    ]
    for name, path in model_files.items():
        summary_lines.append(f"- `{path.name}`: `{path.relative_to(ROOT).as_posix()}` (candidate Test-evaluation model).")
    forecast_model_path = MODEL_DIR / "champion_future_forecast.pkl"
    summary_lines += [
        f"- `champion_future_forecast.pkl`: `{forecast_model_path.relative_to(ROOT).as_posix()}` (refitted on all labeled rows after Test evaluation).",
        "",
        "The forecast model is distinct from the Train + Validation candidate used to report Test metrics; it does not change those Test metrics.",
        "",
        "## Logs and standard directories",
        "",
        f"- `models/lab03/`: {len(standard_models)} `.pkl` files currently present.",
        f"- `logs/lab03/`: {len(run_logs)} `.log` files currently present; this notebook does not create run logs.",
        f"- Run metadata: `{(RUN_DIR / 'run_summary.json').relative_to(ROOT).as_posix()}`.",
        f"- Run reports and figures: `{RUN_DIR.relative_to(ROOT).as_posix()}/`.",
        "",
        "See `docs/lab03/MODEL_SUMMARY_SPEC.md` for field definitions and interpretation.",
    ]
    model_summary_md = "\n".join(summary_lines) + "\n"
    (directory / "model_summary.md").write_text(model_summary_md, encoding="utf-8")
    cv_detail.to_csv(directory / "cv_fold_metrics.csv", index=False)
    write_json(directory / "validation_metrics.json", validation_json)
    write_json(directory / "test_metrics.json", test_json)

write_model_summary(RUN_DIR)
write_model_summary(SUMMARY_ROOT)

cv_summary = []
for name in MODEL_FACTORIES:
    cv_summary.append({
        "model": name,
        "RMSE_mean": float(cv_stats.loc[name, ("RMSE", "mean")]),
        "RMSE_std": float(cv_stats.loc[name, ("RMSE", "std")]),
        "MAE_mean": float(cv_stats.loc[name, ("MAE", "mean")]),
        "R2_mean": float(cv_stats.loc[name, ("R2", "mean")]),
    })
report_model_rows = []
for row in summary_rows:
    report_model_rows.append({
        "model": row["model"],
        "cv_RMSE_mean": row["cv_RMSE_mean"],
        "validation_RMSE": row["validation_RMSE"],
        "test_MAE": row["test_MAE"],
        "test_RMSE": row["test_RMSE"],
        "test_MAPE (%)": row["test_MAPE (%)"],
        "test_R2": row["test_R2"],
        "fit_seconds": row["fit_seconds"],
    })
figure_paths = {
    "hourly_series": FIGURE_DIR / "01_hourly_series.png",
    "temporal_split": FIGURE_DIR / "02_temporal_split.png",
    "test_and_residuals": FIGURE_DIR / "03_test_and_residuals.png",
    "test_model_comparison": FIGURE_DIR / "04_test_model_comparison.png",
    "future_24h_forecast": FIGURE_DIR / "05_future_24h_forecast.png",
    "temporal_profiles": FIGURE_DIR / "06_temporal_profiles.png",
    "daily_trend": FIGURE_DIR / "07_daily_trend.png",
    "walk_forward_cv": FIGURE_DIR / "08_walk_forward_cv.png",
    "test_error_profiles": FIGURE_DIR / "09_test_error_profiles.png",
    "feature_effects": FIGURE_DIR / "10_feature_effects.png",
    "actual_vs_prediction": FIGURE_DIR / "11_actual_vs_prediction.png",
    "test_mae_weekday_hour": FIGURE_DIR / "12_test_mae_weekday_hour.png",
}
relative_figures = {name: path.relative_to(REPORT_ROOT).as_posix()
                    for name, path in figure_paths.items()}
report_payload = {
    "run_id": RUN_ID,
    "status": "SUCCESS",
    "champion": CHAMPION_NAME,
    "summary_text": (
        f"On {ingestion['dataset_name']}, {len(series):,} cleaned hourly observations were used. "
        f"{CHAMPION_NAME} was selected using Validation RMSE "
        f"({validation_summary.loc[CHAMPION_NAME, 'RMSE']:.2f} MW). Its held-out Test RMSE was "
        f"{test_summary.loc[CHAMPION_NAME, 'RMSE']:.2f} MW. A recursive {len(forecast)}-hour forecast was generated."
    ),
    "dataset": {**ingestion, "missing_values_remaining": int(series.isna().sum())},
    "split": {
        "train": {"rows": len(y_train), "start": str(y_train.index.min()), "end": str(y_train.index.max()), "ratio": 0.70},
        "validation": {"rows": len(y_val), "start": str(y_val.index.min()), "end": str(y_val.index.max()), "ratio": 0.15},
        "test": {"rows": len(y_test), "start": str(y_test.index.min()), "end": str(y_test.index.max()), "ratio": 0.15},
    },
    "feature_config": {"lags": list(LAGS), "rolling_windows": list(ROLLING_WINDOWS),
                       "include_calendar_features": True},
    "feature_count": int(X.shape[1]),
    "cv_summary": cv_summary,
    "n_splits": 5,
    "model_rows": report_model_rows,
    "forecast_horizon": len(forecast),
    "forecast_training_rows": int(len(y)),
    "forecast_training_end": str(y.index.max()),
    "figures": {name: str(path) for name, path in figure_paths.items()},
    "figure_relative_paths": {name: f"figures/{path.name}" for name, path in figure_paths.items()},
    "legacy_snapshot": [],
}
run_report_paths = write_report_bundle(report_payload, RUN_DIR)
latest_report_payload = dict(report_payload)
latest_report_payload["figure_relative_paths"] = relative_figures
latest_report_paths = write_report_bundle(latest_report_payload, REPORT_ROOT)

run_summary = {
    "run_id": RUN_ID,
    "dataset": ingestion,
    "feature_count": int(X.shape[1]),
    "split_rows": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "champion_selected_by_validation": CHAMPION_NAME,
    "validation_metrics": validation_summary.loc[CHAMPION_NAME].to_dict(),
    "test_metrics": test_summary.loc[CHAMPION_NAME].drop(labels="validation_RMSE").to_dict(),
    "forecast_horizon_hours": len(forecast),
    "forecast_start": str(forecast.index.min()),
    "forecast_end": str(forecast.index.max()),
    "figure_dir": str(FIGURE_DIR),
    "reports": latest_report_paths,
    "model_summary_dir": str(SUMMARY_ROOT),
}
write_json(RUN_DIR / "run_summary.json", run_summary)
write_json(ROOT / "data" / "processed" / "lab03" / "run_result.json", run_summary)

required_outputs = [
    RUN_DIR / "run_summary.json",
    RUN_DIR / "model_summary.csv",
    RUN_DIR / "model_summary.json",
    RUN_DIR / "model_summary.md",
    RUN_DIR / "cv_fold_metrics.csv",
    RUN_DIR / "validation_metrics.csv",
    RUN_DIR / "test_metrics.csv",
    RUN_DIR / "test_predictions.csv",
    RUN_DIR / "future_24h_forecast.csv",
    SUMMARY_ROOT / "model_summary.csv",
    SUMMARY_ROOT / "model_summary.json",
    SUMMARY_ROOT / "model_summary.md",
    *[Path(path) for path in run_report_paths.values()],
    *[Path(path) for path in latest_report_paths.values()],
    *figure_paths.values(),
]
missing_outputs = [str(path) for path in required_outputs if not path.is_file()]
if missing_outputs:
    raise FileNotFoundError(f"Thiếu artifact sau khi chạy: {missing_outputs}")
expected_model_files = len(MODEL_FACTORIES) + 1
if len(list(MODEL_DIR.glob("*.pkl"))) != expected_model_files:
    raise RuntimeError(f"Số model artifact không đúng: cần {expected_model_files} file trong {MODEL_DIR}")
if len(cv_detail) != 5 * len(MODEL_FACTORIES):
    raise RuntimeError("CV không tạo đủ metric cho 5 fold và toàn bộ model.")

print("Hoàn tất Lab 03.")
print("Champion:", CHAMPION_NAME)
print("Report files:", latest_report_paths)
print("Model summary:", SUMMARY_ROOT)
print("Per-run outputs:", RUN_DIR)
print("Test metrics:", test_summary.loc[CHAMPION_NAME].to_dict())
print("Forecast:", forecast.index.min(), "→", forecast.index.max())

## Câu hỏi thảo luận

1. Vì sao không được dùng Test để chọn champion hoặc tinh chỉnh siêu tham số?
2. So sánh Naive, Linear Regression và các mô hình cây theo Validation/Test. Mô hình nào tổng quát hóa tốt hơn?
3. Vì sao kết quả dự báo một bước trên Test khác với dự báo đệ quy 24 giờ tại một mốc cố định?
4. Lag/rolling nào có thể nắm bắt tính mùa vụ ngày và tuần? Nêu một cách kiểm tra thêm mà không làm rò rỉ dữ liệu tương lai.
5. Sai số theo giờ/thứ cho thấy thời điểm nào khó dự báo hơn? Nêu một giới hạn của feature importance trong dữ liệu có lag tương quan.
6. Vì sao dự báo đệ quy 24 giờ không kèm khoảng bất định và có thể lệch dần khỏi giá trị thật?

Notebook này dùng API Lab03 hiện hành; đây là notebook chạy chính cho dự án.